### Import Modules

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.simplefilter("ignore")
import os, glob
from pathlib import Path

try:
    import utils.process_analysis
    import utils.segment_isc_utils
except ImportError as e:
    print(f"\nSTILL FAILING: {e}")

### Run Sliding Window ISC

In [ ]:
DATA_DIR        = "../data/02_processed_combined_data"
CONDITIONS_FILE = "../data/sub_conditions.csv"
OUTPUT_PLOT    = "../data/04_analysis/dynISC/Figure4_DynISC.png"
file_path = Path(OUTPUT_PLOT)
file_path.parent.mkdir(parents=True, exist_ok=True)

# Constants
FS             = 20
WINDOW_SECONDS = 15
STEP_SECONDS   = 1
WIN_SIZE       = int(WINDOW_SECONDS * FS)
STEP_SIZE      = int(STEP_SECONDS * FS)

signals  = ['Gaze_X', 'Gaze_Y', 'Pupil_Mean', 'Heart_Rate', 'GSR']
signal_labels = {  'Gaze_X': 'Gaze X', 'Gaze_Y': 'Gaze Y',
                   'Pupil_Mean': 'Pupil', 'Heart_Rate': 'Heart Rate', 'GSR': 'GSR' }

pc_subs  = utils.segment_isc_utils.get_subjects_by_condition(CONDITIONS_FILE, 'PC')
pcm_subs = utils.segment_isc_utils.get_subjects_by_condition(CONDITIONS_FILE, 'PCM')

all_results = []

for signal in signals:
    print(f"  Processing {signal}...")

    d_pc = utils.segment_isc_utils.load_physio_data(DATA_DIR, pc_subs, signal)
    d_pcm = utils.segment_isc_utils.load_physio_data(DATA_DIR, pcm_subs, signal)
        
    # 2. COMPUTE ISC FOR GROUP 1 (PC)
    res_pc = utils.process_analysis.sliding_window_split_half_isc(d_pc, WIN_SIZE, STEP_SIZE, FS)
    res_pc['Condition'] = 'Main Movie'
    res_pc['Signal'] = signal
    
    # 3. COMPUTE ISC FOR GROUP 2 (PCM)
    res_pcm = utils.process_analysis.sliding_window_split_half_isc(d_pcm, WIN_SIZE, STEP_SIZE, FS)
    res_pcm['Condition'] = 'Manipulated Movie'
    res_pcm['Signal'] = signal
    
    all_results.append(res_pc)
    all_results.append(res_pcm)

# Combine all results
df_viz = pd.concat(all_results, ignore_index=True)


  Processing Gaze_X...
  Processing Gaze_Y...
  Processing Pupil_Mean...
  Processing Heart_Rate...


### Plot

In [ ]:
BRANCH_TIME = 205
sns.set_theme(font='Arial')
sns.set(style="ticks", context="paper")


fig, axes = plt.subplots(nrows=5, ncols=1, figsize=(6, 9), sharex=True)
#fig.suptitle('      Dynamic ISC')
palette = {'Main Movie': 'blue', 'Manipulated Movie': 'red'} # Blue vs Orange

for i, sig in enumerate(signals):
    ax = axes[i]
    subset = df_viz[df_viz['Signal'] == sig]
    
    # Plot using Seaborn lineplot
    sns.lineplot(data=subset, x='Time_Sec', y='Mean_r', hue='Condition', alpha = 0.7,
                 palette=palette, ax=ax, linewidth=2)
    
    ax.set_title(signal_labels[sig] + ' Dynamic ISC', loc='center', fontsize=12)
    ax.set_ylabel("ISC (r)")
    ax.axhline(0, color='black', lw=0.5)

    ax.axvspan(9, BRANCH_TIME, color='green', alpha=0.1)
    ax.axvspan(BRANCH_TIME, 278, color='red', alpha=0.1)
    ax.axvspan(278, 319, color='green', alpha=0.1)
    
    ax.set_ylim(-0.5, 1.0)
    y_ticks = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    ax.set_yticks(y_ticks)
    y_labels = ["", "", "0", "", "", "", "", "1"]
    ax.set_yticklabels(y_labels, fontsize=10)

    # Only show legend on the first plot
    if i == 0:
        ax.legend(loc='lower center', fontsize=10)
    else:
        ax.get_legend().remove()

axes[-1].set_xlabel("Time (seconds)")
sns.despine()
plt.tight_layout()
plt.savefig(OUTPUT_PLOT, dpi=300)

plt.show()